In [1]:
import os
from d3rlpy.logging import UnifiedFileAdapterFactory

os.environ["D3RLPY_DATASETS_PATH"] = "/gpfs/data/fs72297/jklotz/programming_data/d3rlpy_data"
os.makedirs(os.environ["D3RLPY_DATASETS_PATH"], exist_ok=True)
os.environ["MINARI_DATASETS_PATH"] = "/gpfs/data/fs72297/jklotz/programming_data/d3rlpy_data/minari_data"
os.makedirs(os.environ["MINARI_DATASETS_PATH"], exist_ok=True)

import d3rlpy

dataset, env = d3rlpy.datasets.get_cartpole()
env = gym.make("CartPole-v1",max_episode_steps=200)
seed = 1

dataset_name = "cartpole"
# fix seed
d3rlpy.seed(seed)
d3rlpy.envs.seed_env(env, seed)

if "cartpole" in dataset_name:
    target_return = 200
else:
    raise ValueError("unsupported dataset")

discrete_tacr = d3rlpy.algos.DiscreteTACRConfig(
    batch_size=64,
    actor_learning_rate=1e-4,
    actor_optim_factory=d3rlpy.optimizers.AdamWFactory(
        weight_decay=1e-4,
        clip_grad_norm=0.25,
        lr_scheduler_factory=d3rlpy.optimizers.WarmupSchedulerFactory(
            warmup_steps=10000#10000
        ),
    ),
    actor_encoder_factory=d3rlpy.models.VectorEncoderFactory(
        [128],
        exclude_last_activation=True,
    ),
    observation_scaler=d3rlpy.preprocessing.StandardObservationScaler(),
    position_encoding_type=d3rlpy.PositionEncodingType.SIMPLE,
    context_size=20,
    num_heads=8,
    num_layers=6,
    max_timestep=200,
    compile_graph=True,
    alpha=0.5,
).create(device="cuda:0")

discrete_tacr.fit(
    dataset,
    n_steps=100000,# 100000,
    n_steps_per_epoch=1000,# 1000,
    save_interval=100,
    eval_env=env,
    eval_target_return=target_return,
    experiment_name=f"Discrete_TACR_{dataset_name}_{seed}",
    logger_adapter=UnifiedFileAdapterFactory(),
    n_trials=50,
    eval_gaps=1
)

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


/gpfs/data/fs72297/jklotz/.conda/envs/d3rlpy_dev_requirements_py310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2025-07-27 19:26.19 [info     ] Signatures have been automatically determined. action_signature=Signature(dtype=[dtype('int32')], shape=[(1,)]) observation_signature=Signature(dtype=[dtype('float32')], shape=[(4,)]) reward_signature=Signature(dtype=[dtype('float32')], shape=[(1,)])
2025-07-27 19:26.19 [info     ] Action-space has been automatically determined. action_space=<ActionSpace.DISCRETE: 2>
2025-07-27 19:26.19 [info     ] Action size has been automatically determined. action_size=2


NameError: name 'gym' is not defined